In [2]:
import numpy as np
from ase.units import kB

In [38]:
T = [1]
i = 1
while i < 1000:
    i *= 1.2
    T.append(i)
T = np.array(T)

In [39]:
print(T)

[1.00000000e+00 1.20000000e+00 1.44000000e+00 1.72800000e+00
 2.07360000e+00 2.48832000e+00 2.98598400e+00 3.58318080e+00
 4.29981696e+00 5.15978035e+00 6.19173642e+00 7.43008371e+00
 8.91610045e+00 1.06993205e+01 1.28391846e+01 1.54070216e+01
 1.84884259e+01 2.21861111e+01 2.66233333e+01 3.19479999e+01
 3.83375999e+01 4.60051199e+01 5.52061439e+01 6.62473727e+01
 7.94968472e+01 9.53962166e+01 1.14475460e+02 1.37370552e+02
 1.64844662e+02 1.97813595e+02 2.37376314e+02 2.84851577e+02
 3.41821892e+02 4.10186270e+02 4.92223524e+02 5.90668229e+02
 7.08801875e+02 8.50562250e+02 1.02067470e+03]


In [40]:
print(T[1:])
print(T[:-1])
delta = 1/(kB*T[1:]) - 1/(kB*T[:-1])
print(delta)
np.exp(delta)

[   1.2           1.44          1.728         2.0736        2.48832
    2.985984      3.5831808     4.29981696    5.15978035    6.19173642
    7.43008371    8.91610045   10.69932054   12.83918465   15.40702157
   18.48842589   22.18611107   26.62333328   31.94799994   38.33759992
   46.00511991   55.20614389   66.24737267   79.4968472    95.39621664
  114.47545997  137.37055197  164.84466236  197.81359483  237.3763138
  284.85157656  341.82189187  410.18627025  492.2235243   590.66822915
  708.80187499  850.56224998 1020.67469998]
[  1.           1.2          1.44         1.728        2.0736
   2.48832      2.985984     3.5831808    4.29981696   5.15978035
   6.19173642   7.43008371   8.91610045  10.69932054  12.83918465
  15.40702157  18.48842589  22.18611107  26.62333328  31.94799994
  38.33759992  46.00511991  55.20614389  66.24737267  79.4968472
  95.39621664 114.47545997 137.37055197 164.84466236 197.81359483
 237.3763138  284.85157656 341.82189187 410.18627025 492.2235243
 590.66

array([0.00000000e+000, 0.00000000e+000, 0.00000000e+000, 0.00000000e+000,
       0.00000000e+000, 0.00000000e+000, 4.98867243e-282, 3.81640237e-235,
       4.48103632e-196, 1.61987705e-163, 2.19399202e-136, 8.93365555e-114,
       6.20189396e-095, 3.11724029e-079, 3.78566256e-066, 3.03238903e-055,
       3.69959426e-046, 1.38079248e-038, 2.81907748e-032, 5.11001894e-027,
       1.23126658e-022, 5.52026691e-019, 6.09490205e-016, 2.09317534e-013,
       2.71646850e-011, 1.56676512e-009, 4.59730424e-008, 7.68102841e-007,
       8.02631394e-006, 5.67235565e-005, 2.89380687e-004, 1.12518678e-003,
       3.48888925e-003, 8.95858295e-003, 1.96577029e-002, 3.78393765e-002,
       6.53060339e-002, 1.02910833e-001])

In [21]:
np.sign(23.233545)

1.0

In [15]:
z[1:]**2

array([1.081975  , 0.04896079])

In [ ]:
#this is step A
def position_update(x,v,dt):
    x_new = x + v*dt/2.
    return x_new

#this is step B
def velocity_update(v,F,dt):
    v_new = v + F*dt/2.
    return v_new

def random_velocity_update(v,gamma,kBT,dt):
    R = np.random.normal()
    c1 = np.exp(-gamma*dt)
    c2 = np.sqrt(1-c1*c1)*np.sqrt(kBT)
    v_new = c1*v + R*c2
    return v_new


def baoab(potential, max_time, dt, gamma, kBT, initial_position, initial_velocity,
                                        save_frequency=3, **kwargs ):
    x = initial_position
    v = initial_velocity
    t = 0
    step_number = 0
    positions = []
    velocities = []
    total_energies = []
    save_times = []
    
    while(t<max_time):
        
        # B
        potential_energy, force = potential(x,**kwargs)
        v = velocity_update(v,force,dt)
        
        #A
        x = position_update(x,v,dt)

        #O
        v = random_velocity_update(v,gamma,kBT,dt)
        
        #A
        x = position_update(x,v,dt)
        
        # B
        potential_energy, force = potential(x,**kwargs)
        v = velocity_update(v,force,dt)
        
        if step_number%save_frequency == 0 and step_number>0:
            e_total = .5*v*v + potential_energy

            positions.append(x)
            velocities.append(v)
            total_energies.append(e_total)
            save_times.append(t)
        
        t = t+dt
        step_number = step_number + 1
    
    return save_times, positions, velocities, total_energies   